<h1>RAZ Systems </h1>

# Assignment -- AutoGen Core: Numbers Duel

### Problem Statement

You are building a small multi-agent game using AutoGen's **Core API** (`SingleThreadedAgentRuntime` + `RoutedAgent`) -- the same lower-level building blocks from the lesson notebook, not AgentChat's `Team`.

Two player agents will each be asked to "pick a number between 1 and 10". A third **Judge agent** will collect both picks and decide who wins (the higher number), the same orchestration pattern as the lesson's Rock-Paper-Scissors judge calling two player sub-agents via `self.send_message(...)`.

**Your task:** fill in every `# TODO`. A full solution is at the end -- try not to peek until you've had a go.

**A question to investigate once it's working:** run the duel 8-10 times in a row. Do the numbers look evenly spread across 1-10, or does one number keep showing up suspiciously often? This is the exact same phenomenon you just diagnosed in the lesson's rock-paper-scissors demo -- LLMs sampled at a given temperature are not a fair dice roll, they're drawing from a *learned* (and often skewed) distribution. Asking an LLM to "pick a random number 1-10" is a famous example of this -- many models disproportionately say 7.

In [ ]:
# --- Imports ---
from dataclasses import dataclass
from autogen_core import AgentId, MessageContext, RoutedAgent, message_handler
from autogen_core import SingleThreadedAgentRuntime
from autogen_agentchat.agents import AssistantAgent
from autogen_agentchat.messages import TextMessage
from autogen_ext.models.openai import OpenAIChatCompletionClient
from dotenv import load_dotenv

load_dotenv(override=True)

### First concept: Define the Message object

Same as the lesson -- a simple dataclass with one field.

**TODO:** define a `Message` dataclass with a single `content: str` field.

In [ ]:
# --- TODO ---
@dataclass
class Message:
    ___: ___   # TODO: content: str

### Second concept: An LLM-delegating RoutedAgent (one per player)

Each player is a `RoutedAgent` that, on receiving a message, hands it off to its own internal `AssistantAgent` -- the same delegation pattern as the lesson's `Player1Agent`/`Player2Agent`.

**TODO:** complete `Player1Agent` and `Player2Agent`. Each needs:
- its own `OpenAIChatCompletionClient`
- its own `AssistantAgent` delegate
- a `@message_handler` that wraps the incoming `Message` as a `TextMessage`, runs it through the delegate via `on_messages`, and returns a new `Message` with the reply

In [ ]:
# --- TODO ---
class Player1Agent(RoutedAgent):
    def __init__(self, name: str) -> None:
        super().__init__(name)
        model_client = ___  # TODO: OpenAIChatCompletionClient(model="gpt-4o-mini", temperature=1.0)
        self._delegate = ___  # TODO: AssistantAgent(name, model_client=model_client)

    @message_handler
    async def handle_my_message_type(self, message: Message, ctx: MessageContext) -> Message:
        text_message = ___  # TODO: wrap message.content in a TextMessage, source="user"
        response = await ___  # TODO: self._delegate.on_messages([text_message], ctx.cancellation_token)
        return Message(content=___)  # TODO: response.chat_message.content


class Player2Agent(RoutedAgent):
    def __init__(self, name: str) -> None:
        super().__init__(name)
        model_client = ___  # TODO
        self._delegate = ___  # TODO

    @message_handler
    async def handle_my_message_type(self, message: Message, ctx: MessageContext) -> Message:
        text_message = ___  # TODO
        response = await ___  # TODO
        return Message(content=___)  # TODO

### Third concept: The Judge agent -- an orchestrator calling two sub-agents

This is the key new pattern from the lesson: one `RoutedAgent` calling *other registered agents* via `self.send_message(message, agent_id)`, collecting both replies, then using its own delegate to reason about the result.

**TODO:**
1. Write an `instruction` string asking for a single number between 1 and 10, digits only
2. Build `AgentId`s for `"player1"` and `"player2"`
3. Send the instruction to both, collect both replies
4. Build a judgement prompt and have the delegate decide the winner

In [ ]:
# --- TODO ---
JUDGE = "You are judging a number-picking duel. The players picked these numbers:\n"

class NumbersDuelAgent(RoutedAgent):
    def __init__(self, name: str) -> None:
        super().__init__(name)
        model_client = OpenAIChatCompletionClient(model="gpt-4o-mini", temperature=1.0)
        self._delegate = AssistantAgent(name, model_client=model_client)

    @message_handler
    async def handle_my_message_type(self, message: Message, ctx: MessageContext) -> Message:
        instruction = ___  # TODO: ask for a single number 1-10, digits only, no words
        outgoing = Message(content=instruction)

        inner_1 = ___  # TODO: AgentId("player1", "default")
        inner_2 = ___  # TODO: AgentId("player2", "default")

        response1 = await ___  # TODO: self.send_message(outgoing, inner_1)
        response2 = await ___  # TODO: self.send_message(outgoing, inner_2)

        result = f"Player 1: {response1.content}\nPlayer 2: {response2.content}\n"
        judgement_prompt = f"{JUDGE}{result}Who wins -- higher number wins. State the winner and why."
        judgement_message = TextMessage(content=judgement_prompt, source="user")

        response = await self._delegate.on_messages([judgement_message], ctx.cancellation_token)
        return Message(content=result + response.chat_message.content)

### Fourth concept: Register, start, send a message, stop

**TODO:** register all three agents under the names `"player1"`, `"player2"`, and `"numbers_duel"`, then start the runtime.

In [ ]:
# --- TODO ---
runtime = SingleThreadedAgentRuntime()
await Player1Agent.register(runtime, ___, lambda: Player1Agent(___))  # TODO: "player1"
await Player2Agent.register(runtime, ___, lambda: Player2Agent(___))  # TODO: "player2"
await NumbersDuelAgent.register(runtime, ___, lambda: NumbersDuelAgent(___))  # TODO: "numbers_duel"
runtime.start()

In [ ]:
# --- TODO: send a message to the duel agent and print the result ---
agent_id = ___  # TODO: AgentId("numbers_duel", "default")
message = Message(content="go")
response = await runtime.send_message(message, agent_id)
print(response.content)

**TODO:** stop and close the runtime.

In [ ]:
# --- TODO ---
await ___  # TODO: runtime.stop()
await ___  # TODO: runtime.close()

---
## What to try next

- Run the duel cell 8-10 times (you'll need to re-register and restart the runtime each time, or wrap it in a loop). Tally the numbers each player picks -- is the spread roughly even across 1-10?
- Try lowering `temperature` to `0.0` for both players. Does the bias get *more* obvious, or does it stay about the same?
- Add a third player and adjust the Judge's instructions accordingly
- Replace the LLM-based number picker with `random.randint(1, 10)` in Python instead, and compare how much more evenly distributed the results become

---
# Solution

No peeking until you've tried it yourself!

In [ ]:
# === Setup ===
from dataclasses import dataclass
from autogen_core import AgentId, MessageContext, RoutedAgent, message_handler
from autogen_core import SingleThreadedAgentRuntime
from autogen_agentchat.agents import AssistantAgent
from autogen_agentchat.messages import TextMessage
from autogen_ext.models.openai import OpenAIChatCompletionClient
from dotenv import load_dotenv

load_dotenv(override=True)

@dataclass
class Message:
    content: str

In [ ]:
# === Player agents ===
class Player1Agent(RoutedAgent):
    def __init__(self, name: str) -> None:
        super().__init__(name)
        model_client = OpenAIChatCompletionClient(model="gpt-4o-mini", temperature=1.0)
        self._delegate = AssistantAgent(name, model_client=model_client)

    @message_handler
    async def handle_my_message_type(self, message: Message, ctx: MessageContext) -> Message:
        text_message = TextMessage(content=message.content, source="user")
        response = await self._delegate.on_messages([text_message], ctx.cancellation_token)
        return Message(content=response.chat_message.content)


class Player2Agent(RoutedAgent):
    def __init__(self, name: str) -> None:
        super().__init__(name)
        model_client = OpenAIChatCompletionClient(model="gpt-4o-mini", temperature=1.0)
        self._delegate = AssistantAgent(name, model_client=model_client)

    @message_handler
    async def handle_my_message_type(self, message: Message, ctx: MessageContext) -> Message:
        text_message = TextMessage(content=message.content, source="user")
        response = await self._delegate.on_messages([text_message], ctx.cancellation_token)
        return Message(content=response.chat_message.content)

In [ ]:
# === Judge / orchestrator agent ===
JUDGE = "You are judging a number-picking duel. The players picked these numbers:\n"

class NumbersDuelAgent(RoutedAgent):
    def __init__(self, name: str) -> None:
        super().__init__(name)
        model_client = OpenAIChatCompletionClient(model="gpt-4o-mini", temperature=1.0)
        self._delegate = AssistantAgent(name, model_client=model_client)

    @message_handler
    async def handle_my_message_type(self, message: Message, ctx: MessageContext) -> Message:
        instruction = (
            "You are playing a number-picking duel. Pick a number between 1 and 10. "
            "Respond with ONLY the digit, no words."
        )
        outgoing = Message(content=instruction)

        inner_1 = AgentId("player1", "default")
        inner_2 = AgentId("player2", "default")

        response1 = await self.send_message(outgoing, inner_1)
        response2 = await self.send_message(outgoing, inner_2)

        result = f"Player 1: {response1.content}\nPlayer 2: {response2.content}\n"
        judgement_prompt = f"{JUDGE}{result}Who wins -- higher number wins. State the winner and why."
        judgement_message = TextMessage(content=judgement_prompt, source="user")

        response = await self._delegate.on_messages([judgement_message], ctx.cancellation_token)
        return Message(content=result + response.chat_message.content)

In [ ]:
# === Register and start ===
runtime = SingleThreadedAgentRuntime()
await Player1Agent.register(runtime, "player1", lambda: Player1Agent("player1"))
await Player2Agent.register(runtime, "player2", lambda: Player2Agent("player2"))
await NumbersDuelAgent.register(runtime, "numbers_duel", lambda: NumbersDuelAgent("numbers_duel"))
runtime.start()

In [ ]:
# === Run the duel ===
agent_id = AgentId("numbers_duel", "default")
message = Message(content="go")
response = await runtime.send_message(message, agent_id)
print(response.content)

In [ ]:
# === Shut down ===
await runtime.stop()
await runtime.close()